# FADC-AV: first image-only 3D experiment
Existing two-channel 3D U-Net; replace enc3's second convolution only.
Four bands, dilation (1,2,3), AdaKern; full validation on all 306 cases every 10 epochs.
Enable GPU and Internet. Attach the SAME dataset version used previously.
Setup saves settings on disk. Later cells reload them and do not depend on old Python variables.
RESUME='auto' resumes last.pth (or legacy last.pt); an explicit path can restore an attached checkpoint.
Keep both last.pth and best.pth when exporting checkpoints. If Kaggle did not preserve working files,
attach saved outputs and set RESUME to last.pth. Checkpoints cannot be recovered from displayed logs.
On restart: run setup and environment checks, then training. Model preflight is skipped when resuming.
Changed timestamps alone do not invalidate new archive signatures. Older reports may need one automatic
array validation scan, with progress; training then continues without manual cell changes.


In [ ]:
from pathlib import Path
import json, subprocess, tempfile
REPO_URL = 'https://github.com/Vemuri-BK/FADC-3D.git'
BRANCH = 'feature/fadc-AV'
EXPECTED_COMMIT = '150df04023f35e222e8b1ccf7ddcb7e772195c59'
CACHE_ROOT = Path('/kaggle/input/datasets/vbk1999/mama-mia-preprocessed-cache-2ch')
OUTPUT = Path('/kaggle/working/outputs/fadc_av_enc3_full_s42')
RESUME = 'auto'  # Or a trusted last.pth path from saved outputs; keep best.pth beside it.
STATE_FILE = Path('/kaggle/working/fadc_av_enc3_session.json')
CONFIG_RELATIVE = 'fadc_av/experiment.json'
assert len(EXPECTED_COMMIT) == 40, 'Set the pinned commit SHA'
assert (CACHE_ROOT / 'train').is_dir() and (CACHE_ROOT / 'val').is_dir(), 'Correct CACHE_ROOT'
def git(*args):
    return subprocess.check_output(['git', *map(str, args)], text=True).strip()
def usable(path):
    try:
        return (git('-C', path, 'remote', 'get-url', 'origin') == REPO_URL
                and git('-C', path, 'rev-parse', 'HEAD') == EXPECTED_COMMIT
                and not git('-C', path, 'status', '--porcelain'))
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False
REPO = Path('/kaggle/working') / ('FADC-AV-' + EXPECTED_COMMIT[:12])
if STATE_FILE.is_file():
    try:
        previous = json.loads(STATE_FILE.read_text())
        if usable(Path(previous['repo'])):
            REPO = Path(previous['repo'])
    except (KeyError, ValueError):
        pass
if not (REPO.exists() and usable(REPO)):
    if REPO.exists():
        print('Preserving existing checkout; creating a clean code folder.')
        REPO = Path(tempfile.mkdtemp(prefix='FADC-AV-clean-', dir='/kaggle/working'))
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', EXPECTED_COMMIT], check=True)
assert usable(REPO)
config = json.loads((REPO / CONFIG_RELATIVE).read_text())
assert config['variant'] == 'fadc_enc3', 'Wrong experiment configuration'
state = dict(repo=str(REPO), config=CONFIG_RELATIVE, cache_root=str(CACHE_ROOT),
             output=str(OUTPUT), resume=RESUME, commit=EXPECTED_COMMIT)
STATE_FILE.write_text(json.dumps(state, indent=2))
print('Setup saved:', STATE_FILE)
print('Experiment:', config['variant'], '| output:', OUTPUT, '| resume:', RESUME)


In [ ]:
from pathlib import Path
import json, importlib.util
STATE_FILE = Path('/kaggle/working/fadc_av_enc3_session.json')
assert STATE_FILE.is_file(), 'Run the setup cell once to restore session settings.'
state = json.loads(STATE_FILE.read_text())
spec = importlib.util.spec_from_file_location('fadc_notebook_runtime', Path(state['repo']) / 'fadc_av/notebook_runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
session = runtime.NotebookSession(state)

session.verify()


In [ ]:
from pathlib import Path
import json, importlib.util
STATE_FILE = Path('/kaggle/working/fadc_av_enc3_session.json')
assert STATE_FILE.is_file(), 'Run the setup cell once to restore session settings.'
state = json.loads(STATE_FILE.read_text())
spec = importlib.util.spec_from_file_location('fadc_notebook_runtime', Path(state['repo']) / 'fadc_av/notebook_runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
session = runtime.NotebookSession(state)

# Optional for resume: prepare() detects existing checkpoints.
session.prepare()


In [ ]:
from pathlib import Path
import json, importlib.util
STATE_FILE = Path('/kaggle/working/fadc_av_enc3_session.json')
assert STATE_FILE.is_file(), 'Run the setup cell once to restore session settings.'
state = json.loads(STATE_FILE.read_text())
spec = importlib.util.spec_from_file_location('fadc_notebook_runtime', Path(state['repo']) / 'fadc_av/notebook_runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
session = runtime.NotebookSession(state)

# Reads saved reports and checkpoints; no report/common/env variables needed.
session.train()


In [ ]:
from pathlib import Path
import json, importlib.util
STATE_FILE = Path('/kaggle/working/fadc_av_enc3_session.json')
assert STATE_FILE.is_file(), 'Run the setup cell once to restore session settings.'
state = json.loads(STATE_FILE.read_text())
spec = importlib.util.spec_from_file_location('fadc_notebook_runtime', Path(state['repo']) / 'fadc_av/notebook_runtime.py')
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
session = runtime.NotebookSession(state)

session.evaluate()
from IPython.display import FileLink, display
for name in ('best.pth', 'last.pth', 'train_log.json', 'train_log.csv', 'evaluation.json'):
    path = session.output / name
    if path.is_file():
        display(FileLink(str(path)))
print('Save/download outputs before ending the Kaggle session.')
